In [1]:
# --- INSTALL DEPENDENCIES ---
# This command installs all necessary libraries for the project.
!pip install gymnasium[atari] ale-py opencv-python matplotlib torch numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 73.1 MB/s eta 0:00:00


In [2]:
# --- LIBRARY IMPORTS ---
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import random
import math
import time
import matplotlib.pyplot as plt
from collections import deque
from typing import Tuple, List, Any, Optional, Dict

In [3]:
# Try to import cv2 and gymnasium to confirm installation
try:
    import cv2
    import gymnasium as gym
    print(f"✅ OpenCV version: {cv2.__version__}")
    print(f"✅ Gymnasium version: {gym.__version__}")
except ImportError as e:
    print(f"⚠️ Error importing libraries: {e}. Please ensure dependencies are installed correctly.")

# Confirm PyTorch device (CPU or GPU)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ PyTorch using device: {DEVICE}")
## Task 1 (Corrected): Atari Environment Wrapper

✅ OpenCV version: 4.11.0
✅ Gymnasium version: 1.1.1
✅ PyTorch using device: cpu


In [4]:
# --- TASK 1 CODE ---
print("--- Initializing Task 1 Components (Environment Wrapper) ---")

class AtariWrapper:
    """A comprehensive wrapper for the Atari environment."""
    def __init__(self, env_name: str, frame_skip: int = 4, num_stacked_frames: int = 4, target_frame_size: Tuple[int, int] = (84, 84)):
        try:
            # Use render_mode='rgb_array' to get observations as numpy arrays without a display window
            self.env = gym.make(env_name, obs_type="rgb", render_mode='rgb_array')
        except (gym.error.NameNotFound, Exception) as e:
            print(f"ERROR: Could not create environment '{env_name}'. Ensure Atari ROMs are installed.")
            print(f"Original error: {e}")
            raise

        self.frame_skip = frame_skip
        self.num_stacked_frames = num_stacked_frames
        self.target_frame_size = target_frame_size
        self.action_space_size = self.env.action_space.n

        self._frames = deque(maxlen=num_stacked_frames)
        print(f"✅ [AtariWrapper] Initialized for '{env_name}'. Action space size: {self.action_space_size}.")

    def _preprocess_frame(self, frame: np.ndarray) -> np.ndarray:
        """Converts a raw frame to a processed float32 numpy array."""
        if frame.ndim == 3:
            frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        frame = cv2.resize(frame, self.target_frame_size, interpolation=cv2.INTER_AREA)
        return frame.astype(np.float32) / 255.0

    def reset(self) -> np.ndarray:
        """Resets the environment and the frame stack."""
        observation, _ = self.env.reset()
        processed_frame = self._preprocess_frame(observation)
        # Fill the stack with the first frame
        for _ in range(self.num_stacked_frames):
            self._frames.append(processed_frame)
        return self._get_stacked_state()

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, dict]:
        """Executes an action with frame skipping."""
        total_reward = 0.0
        last_observation = None
        info = {}
        for _ in range(self.frame_skip):
            observation, reward, terminated, truncated, step_info = self.env.step(action)
            total_reward += reward
            last_observation = observation
            info.update(step_info)
            if terminated or truncated:
                break

        processed_frame = self._preprocess_frame(last_observation)
        self._frames.append(processed_frame)

        return self._get_stacked_state(), total_reward, terminated or truncated, info

    def _get_stacked_state(self) -> np.ndarray:
        """Returns the current stacked state as a numpy array."""
        return np.array(self._frames)

    def close(self):
        self.env.close()
        print("  [AtariWrapper] Environment closed.")

--- Initializing Task 1 Components (Environment Wrapper) ---


In [6]:
## Task 2 (Corrected): Prioritized Experience Replay (PER)
# --- TASK 2 CODE ---
print("--- Initializing Task 2 Components (PER) ---")

class SumTree:
    """A robust SumTree implementation."""
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.tree = np.zeros(2 * capacity - 1)
        self.data = np.zeros(capacity, dtype=object)
        self.write_idx, self.size = 0, 0

    def _propagate(self, idx: int, change: float):
        parent = (idx - 1) // 2
        self.tree[parent] += change
        if parent != 0: self._propagate(parent, change)

    def _retrieve(self, idx: int, s: float) -> int:
        left, right = 2 * idx + 1, 2 * idx + 2
        if left >= len(self.tree): return idx
        return self._retrieve(left, s) if s <= self.tree[left] else self._retrieve(right, s - self.tree[left])

    def total(self) -> float: return self.tree[0]

    def add(self, priority: float, data: Any):
        tree_idx = self.write_idx + self.capacity - 1
        self.data[self.write_idx] = data
        self.update(tree_idx, priority)
        self.write_idx = (self.write_idx + 1) % self.capacity
        self.size = min(self.capacity, self.size + 1)

    def update(self, tree_idx: int, priority: float):
        change = priority - self.tree[tree_idx]
        self.tree[tree_idx] = priority
        self._propagate(tree_idx, change)

    def get(self, s: float) -> Tuple[int, float, Any]:
        idx = self._retrieve(0, s)
        data_idx = idx - self.capacity + 1
        return (idx, self.tree[idx], self.data[data_idx])

class PrioritizedReplayBuffer:
    def __init__(self, capacity: int, alpha: float, beta_start: float, beta_frames: int, epsilon: float):
        self.tree = SumTree(capacity)
        self.alpha, self.beta_start, self.beta_frames, self.epsilon = alpha, beta_start, beta_frames, epsilon
        self.frame_count, self.max_priority = 0, 1.0
        print(f"✅ [PER] Initialized with capacity {capacity}.")

    def _get_beta(self) -> float:
        return min(1.0, self.beta_start + self.frame_count * (1.0 - self.beta_start) / self.beta_frames)

    def push(self, state: Any, action: int, reward: float, next_state: Any, done: bool):
        self.tree.add(self.max_priority, (state, action, reward, next_state, done))

    def sample(self, batch_size: int) -> Optional[Tuple[List[Any], np.ndarray, np.ndarray]]:
        if self.tree.size < batch_size: return None

        experiences, tree_indices, weights = [], np.empty(batch_size, dtype=np.int32), np.empty(batch_size, dtype=np.float32)
        beta, total_p = self._get_beta(), self.tree.total()
        segment = total_p / batch_size

        for i in range(batch_size):
            s = random.uniform(segment * i, segment * (i + 1))
            idx, p, data = self.tree.get(s)
            experiences.append(data)
            tree_indices[i] = idx
            prob = p / total_p if total_p > 0 else 0
            weights[i] = (self.tree.size * prob) ** (-beta) if prob > 0 else 1.0

        if weights.max() > 0: weights /= weights.max()
        return experiences, tree_indices, weights

    def update_priorities(self, tree_indices: np.ndarray, td_errors: np.ndarray):
        priorities = (np.abs(td_errors) + self.epsilon) ** self.alpha
        for idx, p in zip(tree_indices, priorities):
            p_val = float(p)
            self.tree.update(idx, p_val)
            self.max_priority = max(self.max_priority, p_val)

    def increment_frame_count(self): self.frame_count +=1
    def __len__(self) -> int: return self.tree.size

--- Initializing Task 2 Components (PER) ---


In [7]:
# --- TASK 3 CODE ---
print("--- Initializing Task 3 Components (Dueling DQN with Noisy Nets) ---")

class NoisyLinear(nn.Module):
    """A linear layer with factorized Gaussian noise."""
    def __init__(self, in_features: int, out_features: int):
        super(NoisyLinear, self).__init__()
        self.in_features, self.out_features = in_features, out_features

        self.mu_weight = nn.Parameter(torch.empty(out_features, in_features))
        self.sigma_weight = nn.Parameter(torch.empty(out_features, in_features))
        self.mu_bias = nn.Parameter(torch.empty(out_features))
        self.sigma_bias = nn.Parameter(torch.empty(out_features))

        self.register_buffer("epsilon_input", torch.empty(in_features))
        self.register_buffer("epsilon_output", torch.empty(out_features))

        self.reset_parameters()
        self.reset_noise()

    def reset_parameters(self):
        mu_range = 1 / math.sqrt(self.in_features)
        self.mu_weight.data.uniform_(-mu_range, mu_range)
        self.mu_bias.data.uniform_(-mu_range, mu_range)
        self.sigma_weight.data.fill_(0.017) # Value from Rainbow paper
        self.sigma_bias.data.fill_(0.017)

    def reset_noise(self):
        """Generates new noise for the layer."""
        epsilon_in = torch.randn(self.in_features, device=self.mu_weight.device)
        epsilon_out = torch.randn(self.out_features, device=self.mu_weight.device)
        self.epsilon_input.copy_(epsilon_in.sign() * epsilon_in.abs().sqrt())
        self.epsilon_output.copy_(epsilon_out.sign() * epsilon_out.abs().sqrt())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.training:
            # Factorized Gaussian noise
            weight_noise = self.epsilon_output.ger(self.epsilon_input)
            bias_noise = self.epsilon_output
            weight = self.mu_weight + self.sigma_weight * weight_noise
            bias = self.mu_bias + self.sigma_bias * bias_noise
        else:
            weight, bias = self.mu_weight, self.mu_bias
        return F.linear(x, weight, bias)

class DuelingDQN(nn.Module):
    """The Dueling DQN with Noisy Linear layers."""
    def __init__(self, input_shape: Tuple[int, ...], action_dim: int):
        super(DuelingDQN, self).__init__()

        self.cnn_base = nn.Sequential(
            nn.Conv2d(input_shape[0], 16, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, stride=1),
            nn.ReLU()
        )

        flattened_size = self._get_conv_out_size(input_shape)

        # Dueling Heads with Noisy Layers
        self.value_stream = nn.Sequential(
            NoisyLinear(flattened_size, 256),
            nn.ReLU(),
            NoisyLinear(256, 1) # Outputs V(s)
        )

        self.advantage_stream = nn.Sequential(
            NoisyLinear(flattened_size, 256),
            nn.ReLU(),
            NoisyLinear(256, action_dim) # Outputs A(s, a)
        )
        print(f"✅ [DuelingDQN] Initialized. Flattened Conv Output: {flattened_size}")

    def _get_conv_out_size(self, shape: Tuple[int, ...]) -> int:
        with torch.no_grad():
            o = self.cnn_base(torch.zeros(1, *shape))
        return int(np.prod(o.size()))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.cnn_base(x)
        features = features.view(features.size(0), -1)

        values = self.value_stream(features)
        advantages = self.advantage_stream(features)

        # Combine V(s) and A(s,a) to get Q(s,a)
        # Q(s,a) = V(s) + (A(s,a) - mean(A(s,a')))
        q_values = values + (advantages - advantages.mean(dim=1, keepdim=True))
        return q_values

    def reset_noise(self):
        """Resets noise in all NoisyLinear layers."""
        for name, module in self.named_modules():
            if isinstance(module, NoisyLinear):
                module.reset_noise()

--- Initializing Task 3 Components (Dueling DQN with Noisy Nets) ---


In [8]:
## Task 4 (Corrected): Agent Implementation

This corrected `Agent` class orchestrates everything.
- It initializes the `DuelingDQN` policy and target networks.
- It uses the `PrioritizedReplayBuffer`.
- It relies on the network's noisy layers for exploration, so `select_action` is much simpler and does not use epsilon.
- The `update` method correctly handles the learning step.
# --- TASK 4 CODE ---
print("--- Initializing Task 4 Components (Agent) ---")

class Agent:
    def __init__(self, state_shape: Tuple[int, ...], action_size: int, config: Dict[str, Any]):
        self.config = config
        self.device = torch.device(config.get("device", "cpu"))
        self.action_size = action_size

        # Use the DuelingDQN network
        self.policy_net = DuelingDQN(state_shape, action_size).to(self.device)
        self.target_net = DuelingDQN(state_shape, action_size).to(self.device)
        self.update_target_network()
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=config.get("lr", 1e-4))
        self.replay_buffer = PrioritizedReplayBuffer(
            capacity=config.get("buffer_capacity"), alpha=config.get("per_alpha"),
            beta_start=config.get("per_beta_start"), beta_frames=config.get("per_beta_frames"),
            epsilon=config.get("per_epsilon")
        )
        self.training_steps_done = 0
        print(f"✅ [Agent] Initialized on device: {self.device}.")

    def select_action(self, state: np.ndarray) -> int:
        """Selects an action using the policy network (exploration is handled by noisy layers)."""
        with torch.no_grad():
            state_tensor = torch.from_numpy(state).float().unsqueeze(0).to(self.device)
            return self.policy_net(state_tensor).argmax().item()

    def update(self) -> Optional[float]:
        """Performs a learning update step."""
        sampled_data = self.replay_buffer.sample(self.config['batch_size'])
        if sampled_data is None: return None

        experiences, indices, weights = sampled_data
        states, actions, rewards, next_states, dones = zip(*experiences)

        # Convert batch to tensors
        states_t = torch.from_numpy(np.array(states)).float().to(self.device)
        actions_t = torch.from_numpy(np.array(actions)).long().unsqueeze(1).to(self.device)
        rewards_t = torch.from_numpy(np.array(rewards)).float().unsqueeze(1).to(self.device)
        next_states_t = torch.from_numpy(np.array(next_states)).float().to(self.device)
        dones_t = torch.from_numpy(np.array(dones, dtype=np.uint8)).float().unsqueeze(1).to(self.device)
        weights_t = torch.from_numpy(weights).float().unsqueeze(1).to(self.device)

        # --- Loss Calculation (Double DQN) ---
        self.policy_net.train() # Set to train mode for noisy layers
        self.target_net.eval()

        q_s_a = self.policy_net(states_t).gather(1, actions_t)

        with torch.no_grad():
            next_actions = self.policy_net(next_states_t).argmax(dim=1, keepdim=True)
            q_next_target = self.target_net(next_states_t).gather(1, next_actions)
            target_q_s_a = rewards_t + (self.config['gamma'] * q_next_target * (1 - dones_t))

        td_errors = (q_s_a - target_q_s_a).abs()
        loss = (F.smooth_l1_loss(q_s_a, target_q_s_a, reduction='none') * weights_t).mean()

        # --- Optimization ---
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 10.0) # Clip gradients for stability
        self.optimizer.step()

        self.replay_buffer.update_priorities(indices, td_errors.detach().cpu().numpy())
        self.replay_buffer.increment_frame_count()
        self.training_steps_done += 1

        if self.training_steps_done % self.config['target_update_freq'] == 0:
            self.update_target_network()

        return loss.item()

    def update_target_network(self): self.target_net.load_state_dict(self.policy_net.state_dict())
    def save_model(self, path: str): torch.save(self.policy_net.state_dict(), path)

SyntaxError: unterminated string literal (detected at line 6) (<ipython-input-8-3372689200>, line 6)

In [9]:
## Task 5 (Corrected): Training Loop and Evaluation
# --- Training Configuration ---
# These parameters are set for a more serious Phase 2 attempt.
TRAINING_CONFIG = {
    'device': DEVICE,
    'lr': 1e-4, 'gamma': 0.99, 'buffer_capacity': 100000, 'batch_size': 32,
    'target_update_freq': 2000, # In agent training steps
    'per_alpha': 0.6, 'per_beta_start': 0.4, 'per_beta_frames': 500000, 'per_epsilon': 1e-5,
    'initial_replay_size': 10000, 'max_episodes': 2000, # Increased for Phase 2
    'target_score': 30.0, # A reasonable target for Phase 2
    'log_freq': 10, 'train_freq': 4,
    'atari_env_name': "BreakoutNoFrameskip-v4", 'atari_frame_skip': 4,
    'atari_stacked_frames': 4, 'atari_frame_height': 84, 'atari_frame_width': 84,
    'model_save_path': "breakout_dueling_noisy_agent.pth"
}
print(f"✅ Training Configuration loaded.")

✅ Training Configuration loaded.


In [10]:
# --- Main Training Function ---
def train_agent(config: Dict[str, Any]) -> Dict[str, Any]:
    print(f"\n🚀 Starting training...")
    env = AtariWrapper(
        env_name=config['atari_env_name'], frame_skip=config['atari_frame_skip'],
        num_stacked_frames=config['atari_stacked_frames'],
        target_frame_size=(config['atari_frame_height'], config['atari_frame_width'])
    )
    agent = Agent((config['atari_stacked_frames'], config['atari_frame_height'], config['atari_frame_width']), env.action_space_size, config)

    all_rewards, all_losses, mean_scores = [], [], []
    scores_window = deque(maxlen=50)
    total_steps, peak_mean_score = 0, -float('inf')
    start_time, training_started = time.time(), False

    for i_episode in range(1, config['max_episodes'] + 1):
        # Reset noise at the start of each episode for consistent exploration
        agent.policy_net.reset_noise()
        agent.target_net.reset_noise()

        state = env.reset()
        episode_reward = 0.0

        while True:
            action = agent.select_action(state)
            next_state, reward, done, _ = env.step(action)
            agent.replay_buffer.push(state, action, reward, next_state, done)
            state = next_state
            episode_reward += reward
            total_steps += 1

            if len(agent.replay_buffer) > config['initial_replay_size']:
                if not training_started: print(f"  Buffer filled. Starting training..."); training_started = True
                if total_steps % config['train_freq'] == 0:
                    loss = agent.update()
                    if loss is not None: all_losses.append(loss)
            if done: break

        scores_window.append(episode_reward)
        all_rewards.append(episode_reward)
        current_mean = np.mean(scores_window)
        mean_scores.append(current_mean)
        if len(scores_window) == 50: peak_mean_score = max(peak_mean_score, current_mean)

        if i_episode % config['log_freq'] == 0:
            print(f"Ep: {i_episode:4} | Mean(50): {current_mean:6.2f} | Peak(50): {peak_mean_score:.2f} | Steps: {total_steps:7} | Loss(avg): {np.mean(all_losses[-100:]):.4f}")

        if current_mean >= config['target_score'] and len(scores_window) >= 50:
            print(f"\n🎉 Target score of {config['target_score']:.1f} reached in {i_episode} episodes!")
            break

    env.close()
    agent.save_model(config['model_save_path'])
    return {"rewards": all_rewards, "means": mean_scores, "losses": all_losses}

In [11]:
# --- Visualization Function ---
def plot_training_results(metrics: Dict[str, Any], config: Dict[str, Any]):
    print("\n📊 Generating training visualization...")
    rewards, means, losses = metrics['rewards'], metrics['means'], metrics['losses']
    fig, axs = plt.subplots(2, 2, figsize=(18, 14), constrained_layout=True)
    fig.suptitle(f"Dueling DQN Training Results - {config['atari_env_name']}", fontsize=20, weight='bold')

    axs[0, 0].set_title('Episode Rewards', fontsize=16)
    axs[0, 0].plot(rewards, alpha=0.6, color='deepskyblue')
    if len(rewards) >= 50:
        axs[0, 0].plot(np.convolve(rewards, np.ones(50)/50, mode='valid'), color='dodgerblue', lw=2.5, label='Rolling Mean (50 ep)')
    axs[0, 0].legend()

    axs[0, 1].set_title('Mean Score (50 Episodes)', fontsize=16)
    axs[0, 1].plot(means, color='forestgreen', lw=2.5)
    axs[0, 1].axhline(y=config['target_score'], color='red', linestyle='--', label=f"Target ({config['target_score']})", lw=2)
    axs[0, 1].legend()

    axs[1, 0].set_title('Training Loss', fontsize=16)
    if losses:
        axs[1, 0].plot(losses, alpha=0.6, color='mediumorchid')
        if len(losses) >= 100: axs[1, 0].plot(np.convolve(losses, np.ones(100)/100, mode='valid'), color='darkmagenta', lw=2.5, label='Rolling Mean (100 updates)')
        axs[1, 0].set_yscale('log'); axs[1, 0].legend()
    else:
        axs[1, 0].text(0.5, 0.5, 'No Loss Data', ha='center', va='center')

    axs[1, 1].set_title('Reward Distribution', fontsize=16)
    axs[1, 1].hist(rewards, bins=30, color='coral', edgecolor='saddlebrown', alpha=0.75)
    axs[1, 1].axvline(np.mean(rewards), color='firebrick', linestyle='--', lw=2, label=f'Mean: {np.mean(rewards):.2f}')
    axs[1, 1].legend()

    for ax in axs.flat: ax.grid(True, linestyle=':', alpha=0.6)
    plt.savefig("training_results_phase2.png")
    plt.show()

In [12]:
# --- MAIN EXECUTION BLOCK ---
if __name__ == "__main__":
    print("\n===================================================")
    print("    Assignment 02: Phase 1 & 2 RUNNER    ")
    print("===================================================\n")

    # This will run the entire training process.
    # Be aware: with the current Phase 2 config, this will take a long time to run.
    # You can adjust 'max_episodes' in TRAINING_CONFIG for a shorter test run.
    training_results = train_agent(TRAINING_CONFIG)

    if training_results and training_results.get("rewards"):
        plot_training_results(training_results, TRAINING_CONFIG)

    print("\n===================================================")
    print("          ASSIGNMENT SCRIPT END           ")
    print("===================================================")


    Assignment 02: Phase 1 & 2 RUNNER    


🚀 Starting training...
ERROR: Could not create environment 'BreakoutNoFrameskip-v4'. Ensure Atari ROMs are installed.
Original error: Environment `BreakoutNoFrameskip` doesn't exist.


NameNotFound: Environment `BreakoutNoFrameskip` doesn't exist.